##### Copyright 2018 The TF-Agents Authors.

##### Some enhancements and added info was added by guyk1971

### Get Started
<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/agents/blob/master/tf_agents/colabs/5_replay_buffers_tutorial.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/agents/blob/master/tf_agents/colabs/5_replay_buffers_tutorial.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>


In [1]:
# If you haven't installed tf-agents or gym yet, run:
try:
    %tensorflow_version 2.x
except:
    pass
# !pip install tf-agents-nightly
# !pip install gym

import os
import sys
print('inserting the following to path',os.path.abspath('../..'))
sys.path.insert(0,os.path.abspath('../..'))
print(sys.path)

inserting the following to path /home/guy/workspace/study/git/agents
['/home/guy/workspace/study/git/agents', '/home/guy/anaconda3/envs/rltf20/lib/python36.zip', '/home/guy/anaconda3/envs/rltf20/lib/python3.6', '/home/guy/anaconda3/envs/rltf20/lib/python3.6/lib-dynload', '', '/home/guy/.local/lib/python3.6/site-packages', '/home/guy/anaconda3/envs/rltf20/lib/python3.6/site-packages', '/home/guy/anaconda3/envs/rltf20/lib/python3.6/site-packages/IPython/extensions', '/home/guy/.ipython']


### Imports

In [2]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import tensorflow as tf
import numpy as np

from tf_agents import specs
from tf_agents.agents.dqn import dqn_agent
from tf_agents.drivers import dynamic_step_driver
from tf_agents.environments import suite_gym
from tf_agents.environments import tf_py_environment
from tf_agents.networks import q_network
from tf_agents.replay_buffers import py_uniform_replay_buffer
from tf_agents.replay_buffers import tf_uniform_replay_buffer
from tf_agents.specs import tensor_spec
from tf_agents.trajectories import time_step

# my import to control gpu usage
from tf_agents.utils.my_utils import set_gpu_device


tf.compat.v1.enable_v2_behavior()
set_gpu_device('0')

1 Physical GPUs, 1 Logical GPU


# Introduction

Reinforcement learning algorithms use replay buffers to store trajectories of experience when executing a policy in an environment. During training, replay buffers are be queried for a subset of the trajectories (either a sequential subset or a sample) to "replay" the agent's experience.

In this colab, we explore two types of replay buffers: python-backed and tensorflow-backed, sharing a common API. In the following sections, we describe the API, each of the buffer implementations and how to use them during data collection training.



# Replay Buffer API
The Replay Buffer class has the following definition and methods: (see `tf_agents/replay_buffes/replay_buffer.py`)

```python
class ReplayBuffer(tf.Module):
  """Abstract base class for TF-Agents replay buffer."""

  def __init__(self, data_spec, capacity):
    """Initializes the replay buffer.

    Args:
      data_spec: A spec or a list/tuple/nest of specs describing
        a single item that can be stored in this buffer
      capacity: number of elements that the replay buffer can hold.
    """

  @property
  def data_spec(self):
    """Returns the spec for items in the replay buffer."""

  @property
  def capacity(self):
    """Returns the capacity of the replay buffer."""

  def add_batch(self, items):
    """Adds a batch of items to the replay buffer."""

  def get_next(self,
               sample_batch_size=None,
               num_steps=None,
               time_stacked=True):
    """Returns an item or batch of items from the buffer."""

  def as_dataset(self,
                 sample_batch_size=None,
                 num_steps=None,
                 num_parallel_calls=None):
    """Creates and returns a dataset that returns entries from the buffer."""


  def gather_all(self):
    """Returns all the items in buffer."""
    return self._gather_all()

  def clear(self):
    """Resets the contents of replay buffer"""

```

Note that when the replay buffer object is initialized, it requires the `data_spec` of the elements that it will store. This spec corresponds to the `TensorSpec` of trajectory elements that will be added to the buffer. This spec is usually acquired by looking at an agent's `agent.collect_data_spec` which defines the shapes, types, and structures expected by the agent when training (more on that later)

# TFUniformReplayBuffer

`TFUniformReplayBuffer` is the most commonly used replay buffer in TF-Agents, thus we will use it in our tutorial here. In `TFUniformReplayBuffer` the backing buffer storage is done by tensorflow variables and thus is part of the compute graph. 

The buffer stores batches of elements and has a maximum capacity `max_length` elements per batch segment. Thus, the total buffer capacity is `batch_size` x `max_length` elements. The elements stored in the buffer must all have a matching data spec. When the replay buffer is used for data collection, the spec is the agent's collect data spec.

## Creating the buffer:
To create a `TFUniformReplayBuffer` we pass in:
1. the spec of the data elements that the buffer will store
2. the `batch size` corresponding to the batch size of the buffer 
3. the `max_length` number of elements per batch segment

Here is an example of creating a `TFUniformReplayBuffer` with sample data specs, `batch_size` 32 and `max_length` 1000.







In [ ]:
data_spec =  (
        tf.TensorSpec([3], tf.float32, 'action'),
        (
            tf.TensorSpec([5], tf.float32, 'lidar'),
            tf.TensorSpec([3, 2], tf.float32, 'camera')
        )
)

batch_size = 32
max_length = 1000

replay_buffer = tf_uniform_replay_buffer.TFUniformReplayBuffer(
    data_spec,
    batch_size=batch_size,
    max_length=max_length)

## Writing to the buffer:
To add elements to the replay buffer, we use the `add_batch(items)` method where `items` is a list/tuple/nest of tensors representing the batch of items to be added to the buffer. Each element of `items` must have an outer dimension equal `batch_size` and the remaining dimensions must adhere to the data spec of the item (same as the data specs passed to the replay buffer constructor). 

<font color='red'> this is unclear </font>


according to the documentation of tf_uniform_replay_buffer.py:
```
    The TFUniformReplayBuffer stores episodes in `B == batch_size` blocks of
    size `L == max_length`, with total frame capacity
    `C == L * B`.  Storage looks like:

    block1 ep1 frame1
               frame2
           ...
           ep2 frame1
               frame2
           ...
           <L frames total>
    block2 ep1 frame1
               frame2
           ...
           ep2 frame1
               frame2
           ...
           <L frames total>
    ...
    blockB ep1 frame1
               frame2
           ...
           ep2 frame1
               frame2
           ...
           <L frames total>


    Multiple episodes may be stored within a given block, up to `max_length`
    frames total.  In practice, new episodes will overwrite old ones as the
    block rolls over its `max_length`.
```

Here's an example of adding a batch of items 


In [ ]:
action = tf.constant(1 * np.ones(data_spec[0].shape.as_list(), dtype=np.float32))
lidar = tf.constant(2 * np.ones(data_spec[1][0].shape.as_list(), dtype=np.float32))
camera = tf.constant(3 * np.ones(data_spec[1][1].shape.as_list(), dtype=np.float32))
  
values = (action, (lidar, camera))   
values_batched = tf.nest.map_structure(lambda t: tf.stack([t] * batch_size),values)
  
replay_buffer.add_batch(values_batched)


## Reading from the buffer

There are three ways to read data from the `TFUniformReplayBuffer`:

1. `get_next()` - returns one sample from the buffer. The sample batch size and number of timesteps returned can be specified via arguments to this method.
2. `as_dataset()` - returns the replay buffer as a `tf.data.Dataset`. One can then create a dataset iterator and iterate through the samples of the items in the buffer.
3. `gather_all()` - returns all the items in the buffer as a Tensor with shape `[batch, time, data_spec]`

Below are examples of how to read from the replay buffer using each of these methods:



<font color='red'> **TODO**: create a specific values_batch s.t. I can understand how data is saved in the buffer </font>

In [ ]:
# add more items to the buffer before reading
for _ in range(5):
    replay_buffer.add_batch(values_batched)

# Get one sample from the replay buffer with batch size 10 and 1 timestep:
sample = replay_buffer.get_next(sample_batch_size=10, num_steps=1)

# Convert the replay buffer to a tf.data.Dataset and iterate through it
dataset = replay_buffer.as_dataset(
    sample_batch_size=4,    # each batch we sample will include 4 trajectories? are these necessarily full episodes ? 
    num_steps=2)   # no, this is the number of steps in each trajectory

iterator = iter(dataset)
print("Iterator trajectories:")
trajectories = []
for _ in range(3):
    t, _ = next(iterator)
    trajectories.append(t)
  
print(tf.nest.map_structure(lambda t: t.shape, trajectories))

# Read all elements in the replay buffer:
trajectories3 = replay_buffer.gather_all()

print("Trajectories from gather all:")
print(tf.nest.map_structure(lambda t: t.shape, trajectories))

## Further exploring the replay buffer mode of operation
Tryingto understand the idea of `batch_size` parameter for the replay buffer

In [20]:
ds =  (tf.TensorSpec([3], tf.float32, 'action'),
       tf.TensorSpec([5], tf.float32, 'lidar'))

batch_size = 4
max_length = 10
rb = tf_uniform_replay_buffer.TFUniformReplayBuffer(
    ds,
    batch_size=batch_size,
    max_length=max_length)

In [10]:
def create_values(act,lid):
    action = tf.constant(act * np.ones(ds[0].shape.as_list(), dtype=np.float32))
    lidar = tf.constant(lid * np.ones(ds[1].shape.as_list(), dtype=np.float32))
    values = (action, lidar)
    return values

def create_values_batched(base=0, batch_size=32):
    actions=[]
    lidars=[]
    for i in range(batch_size):
        a,l=create_values(base+i*10+1,base+i*10+2)
        actions.append(a)
        lidars.append(l)
    values_batched=(tf.stack(actions),tf.stack(lidars))
    return values_batched
    

In [21]:
# crating a batch of samples (s,a) to observe  the structure
vs=create_values_batched(0,4)
vs

(<tf.Tensor: id=1576, shape=(4, 3), dtype=float32, numpy=
 array([[ 1.,  1.,  1.],
        [11., 11., 11.],
        [21., 21., 21.],
        [31., 31., 31.]], dtype=float32)>,
 <tf.Tensor: id=1577, shape=(4, 5), dtype=float32, numpy=
 array([[ 2.,  2.,  2.,  2.,  2.],
        [12., 12., 12., 12., 12.],
        [22., 22., 22., 22., 22.],
        [32., 32., 32., 32., 32.]], dtype=float32)>)

In [24]:
# writing to the replay buffer
rb.clear()
# fill in the buffer with 5 batches of data
for i in range(5):
    vs=create_values_batched(i*100,4)
    rb.add_batch(vs)

In [47]:
# note that we now fetch from the buffer with a different batch size : sample_batch_size which has the expected meaning we're used to
# what if we ask for num_steps that are longer than the episode ? 
s = rb.get_next(sample_batch_size=5, num_steps=5)
s

((<tf.Tensor: id=3372, shape=(5, 5, 3), dtype=float32, numpy=
  array([[[ 21.,  21.,  21.],
          [121., 121., 121.],
          [221., 221., 221.],
          [321., 321., 321.],
          [421., 421., 421.]],
  
         [[ 11.,  11.,  11.],
          [111., 111., 111.],
          [211., 211., 211.],
          [311., 311., 311.],
          [411., 411., 411.]],
  
         [[  1.,   1.,   1.],
          [101., 101., 101.],
          [201., 201., 201.],
          [301., 301., 301.],
          [401., 401., 401.]],
  
         [[ 11.,  11.,  11.],
          [111., 111., 111.],
          [211., 211., 211.],
          [311., 311., 311.],
          [411., 411., 411.]],
  
         [[ 11.,  11.,  11.],
          [111., 111., 111.],
          [211., 211., 211.],
          [311., 311., 311.],
          [411., 411., 411.]]], dtype=float32)>,
  <tf.Tensor: id=3373, shape=(5, 5, 5), dtype=float32, numpy=
  array([[[ 22.,  22.,  22.,  22.,  22.],
          [122., 122., 122., 122., 122.],
       

It looks like the `batch_size` parameter of the replay buffer is somewhat confusing. its related to the way samples are fed into and arranged in the buffer. its not clear what is the usage that required this arrangement. what optimization does it do ? 

If we look at the various gym environments and the way the agents that are trained are using the buffers, we observe that in most (if not all) cases of gym environments, we use batch_size = 1.

See for example how DQN agent is configured:
```
# define the environment
tf_env = tf_py_environment.TFPyEnvironment(suite_gym.load(env_name))    

# define the q network
q_net = q_network.QNetwork(tf_env.observation_spec(),tf_env.action_spec(),fc_layer_params=fc_layer_params)
          
# define the agent
tf_agent = dqn_agent.DqnAgent(
    tf_env.time_step_spec(),
    tf_env.action_spec(),
    q_network=q_net,
    epsilon_greedy=epsilon_greedy,
    n_step_update=n_step_update,
    target_update_tau=target_update_tau,
    target_update_period=target_update_period,
    optimizer=tf.compat.v1.train.AdamOptimizer(learning_rate=learning_rate),
    td_errors_loss_fn=common.element_wise_squared_loss,
    gamma=gamma,
    reward_scale_factor=reward_scale_factor,
    gradient_clipping=gradient_clipping,
    debug_summaries=debug_summaries,
    summarize_grads_and_vars=summarize_grads_and_vars,
    train_step_counter=global_step)

# define the replay buffer 
replay_buffer = tf_uniform_replay_buffer.TFUniformReplayBuffer(
    data_spec=tf_agent.collect_data_spec,
    batch_size=tf_env.batch_size,
    max_length=replay_buffer_capacity)

# next we define the driver (sampler) etc... 

```
Now, if we go over most (if not all) of gym environments as done for example in the following cell, we'll notice that in all cases we get `tf_env.batch_size=1` meaning that we feed single time steps and not batches of them.
There are probaly some environments (maybe atari ?) that provide batches of samples ? what is the usage ?



In [ ]:
env = suite_gym.load('Acrobot-v1')
tf_env = tf_py_environment.TFPyEnvironment(env)
tf_env.batch_size

When we saved 5 batches of samples, each insertion represnts a time stamp. so if we assume that the data spec includes scalars for action and lidar and we look at the action only, then the buffer will look like that: 

|t0 |t1 |t2 |t3 |t4 |
|---|---|---|---|---|
| 1 |101|201|301|401|
|11 |111|211|311|411|
|21 |121|221|321|421|
|31 |131|231|331|431|


when we want to pull sample from the buffer, it samples trajectories whose length is no longer than 5 (t0-t4) that's why if we ask :
`rb.get_next(sample_batch_size=5, num_steps=6)`   
we'll get an error.


# PyUniformReplayBuffer
`PyUniformReplayBuffer`  has the same functionaly as the `TFUniformReplayBuffer` but instead of tf variables, it's data is stored in numpy arrays. This buffer can be used for out-of-graph data collection. Having the backing storage in numpy may make it easier for some applicaitons to do data manipulation (such as indexing for updating priorities) without using Tensorflow variables. However, this implementation won't have the benefit of graph optimizations with Tensorflow. 

Below is an example of instantiating a `PyUniformReplayBuffer` from the agent's policy trajectory specs:

In [ ]:
replay_buffer_capacity = 1000*32 # same capacity as the TFUniformReplayBuffer

py_replay_buffer = py_uniform_replay_buffer.PyUniformReplayBuffer(
    capacity=replay_buffer_capacity,
    data_spec=tensor_spec.to_nest_array_spec(data_spec))

# Using replay buffers during training
Now that we know how to created a replay buffer, write items to it and read from it, we can use it to store trajectories during training of our agents. 

## Data collection
First, let's look at how to use the replay buffer during data collection.

In TF-Agents we use a `Driver` (see the Driver tutorial for more details) to collect experience in an environment. To use a `Driver`, we specify an `Observer` that is a function for the `Driver` to execute when it receives a trajectory. 

Thus, to add trajectory elements to the replay buffer, we add an observer that calls `add_batch(items)` to add a (batch of) items on the replay buffer. 

Below is an example of this with `TFUniformReplayBuffer`. We first create an environment, a network and an agent. Then we create a `TFUniformReplayBuffer`. Note that the specs of the trajectory elements in the replay buffer are equal to the agent's collect data spec. We then set its `add_batch` method as the observer for the driver that will do the data collect during our training:



In [3]:
env = suite_gym.load('CartPole-v0')
tf_env = tf_py_environment.TFPyEnvironment(env)

q_net = q_network.QNetwork(
    tf_env.time_step_spec().observation,
    tf_env.action_spec(),
    fc_layer_params=(100,))

agent = dqn_agent.DqnAgent(
    tf_env.time_step_spec(),
    tf_env.action_spec(),
    q_network=q_net,
    optimizer=tf.compat.v1.train.AdamOptimizer(0.001))

replay_buffer_capacity = 1000

replay_buffer = tf_uniform_replay_buffer.TFUniformReplayBuffer(
    agent.collect_data_spec,
    batch_size=tf_env.batch_size,     # this value is typically =1 
    max_length=replay_buffer_capacity)

# Add an observer that adds to the replay buffer:
replay_observer = [replay_buffer.add_batch]

collect_steps_per_iteration = 10
collect_op = dynamic_step_driver.DynamicStepDriver(
  tf_env,
  agent.collect_policy,
  observers=replay_observer,
  num_steps=collect_steps_per_iteration).run()

## Reading data for a train step

After adding trajectory elements to the replay buffer, we can read batches of trajectories from the replay buffer to use as input data for a train step.

Here is an example of how to train on trajectories from the replay buffer in a training loop: 

In [4]:
# Read the replay buffer as a Dataset,
# read batches of 4 elements, each with 2 timesteps:
dataset = replay_buffer.as_dataset(
    sample_batch_size=4,
    num_steps=2)

iterator = iter(dataset)

num_train_steps = 10

for _ in range(num_train_steps):
    trajectories, _ = next(iterator)
    loss = agent.train(experience=trajectories)



In [5]:
trajectories[0]

<tf.Tensor: id=4003, shape=(4, 2), dtype=int32, numpy=
array([[1, 1],
       [1, 1],
       [1, 1],
       [1, 1]], dtype=int32)>